# MFF OpenScience Validation — Analysis Notebook

> Notebook **scheletro riproducibile** per l'analisi delle 5 ipotesi H1-H5.
>
> Esegui dopo aver scaricato `DATA/submissions.csv` aggiornato dal sito.
>
> **Status**: stub iniziale (2026-06-03). Le celle di test statistici concreti vengono popolate dopo che N raggiunge la soglia descritta in PROTOCOL.md §3.

## 1. Setup e caricamento dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
df = pd.read_csv('DATA/submissions.csv')
print(f'Loaded {len(df)} submissions')
df.head()

In [ ]:
df_valid = df[df['status'] == 'valid'].copy() if 'status' in df.columns else df.copy()
print(f'Valid: {len(df_valid)} / {len(df)} ({100*len(df_valid)/max(len(df),1):.1f}%)')
if len(df_valid) > 0 and 'delta' not in df_valid.columns:
    df_valid['delta'] = df_valid['rating_mff'] - df_valid['rating_baseline']

## 2. EDA

In [ ]:
if len(df_valid) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_valid['rating_baseline'].hist(ax=axes[0], bins=5, edgecolor='black')
    axes[0].set_title('Rating distribution — Baseline (vanilla)')
    axes[0].set_xlabel('Stars (1-5)')
    df_valid['rating_mff'].hist(ax=axes[1], bins=5, edgecolor='black', color='#D4774A')
    axes[1].set_title('Rating distribution — MFF')
    axes[1].set_xlabel('Stars (1-5)')
    plt.tight_layout()
    plt.show()
else:
    print('Nessuna submission valida ancora.')

## 3. H1 — Riduzione hallucinazioni

Mann-Whitney U su categoria 'fatti'. Target Cohen's d ≥ 0.3.

In [ ]:
df_h1 = df_valid[df_valid['prompt_category'] == 'fatti'] if 'prompt_category' in df_valid.columns else df_valid.iloc[:0]
if len(df_h1) < 30:
    print(f'⚠ N={len(df_h1)} insufficient. Need ≥ 30 per sub-analisi (PROTOCOL.md §3).')
else:
    u, p = stats.mannwhitneyu(df_h1['rating_mff'], df_h1['rating_baseline'], alternative='greater')
    pooled = np.sqrt((df_h1['rating_baseline'].std()**2 + df_h1['rating_mff'].std()**2) / 2)
    d = (df_h1['rating_mff'].mean() - df_h1['rating_baseline'].mean()) / pooled if pooled > 0 else 0
    print(f'H1 — U={u:.1f}, p={p:.4f}, Cohen d={d:.3f}')
    print(f'H1 — Supported: {p < 0.05 and d >= 0.3}')

## 4-7. H2, H3, H4, H5 (placeholder)

Popolati quando N ≥ 300. Vedi PROTOCOL.md §4 per dettagli analisi:
- H2: ECE estrazione etichette 🟢🔵🟡🔴 + ground truth annotators
- H3: TOST equivalence ±0.2 su 'calcoli' + 'creativita'
- H4: ANOVA mff_used × language
- H5: linear mixed-effects con ai_provider random effect

## 8. Export risultati

In [ ]:
import json, datetime
results = {
    'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'n_total': len(df),
    'n_valid': len(df_valid),
    'analysis_version': '0.1.0-stub',
    'hypotheses': {
        'H1': {'status': 'pending_data', 'note': 'Need N≥175 to test'},
        'H2': {'status': 'pending_data', 'note': 'Need annotators'},
        'H3': {'status': 'pending_data'},
        'H4': {'status': 'pending_data'},
        'H5': {'status': 'pending_data'},
    },
}
with open('DATA/analysis_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✓ Results exported to DATA/analysis_results.json')